# ResNet50 — Colab

Çalışma zamanı: **T4 GPU**. Menü: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4.

Akış: ResNet50 artırmalı eğitim → test ölçümü → kazanan seçimi → kazananın artırmasız koşusu → şekiller.

`veri_isleme.py` çalıştırma. `data/splits.csv` kilitlidir. Ayarlar `configs/ortak.yaml` dosyasından okunur, değiştirme.

Her bölümün sonunda sonuçlar zip olarak iner; oturum kapanırsa emek kaybolmaz.

In [ ]:
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'GPU kapalı. Çalışma zamanını T4 yapıp yeniden başlat.'

In [ ]:
%cd /content
!rm -rf Final-project
!git clone https://github.com/bernademirer/Final-project.git
%cd /content/Final-project
!git log --oneline -3
!ls src configs data results

In [ ]:
!pip -q install -r requirements.txt datasets

In [ ]:
from pathlib import Path
import pandas as pd
from datasets import load_dataset

splits = pd.read_csv('data/splits.csv')
assert len(splits) == 1457
assert (splits.kume == 'test').sum() == 292
assert splits.file_name.nunique() == 1457

out = Path('data/images')
out.mkdir(parents=True, exist_ok=True)
need = set(splits.file_name) - {p.name for p in out.glob('*.jpg')}
if need:
    for row in load_dataset('Layered-Labs/ACNE04')['train']:
        if row['file_name'] in need:
            row['image'].save(out / row['file_name'])
print('jpg', len(list(out.glob('*.jpg'))))
assert len(list(out.glob('*.jpg'))) == 1457

## 1. ResNet50 artırmalı eğitim ve test

Eğitim T4 üzerinde yaklaşık 30–60 dakika sürer. Erken durdurma doğrulama macro-F1'i 5 epoch iyileşmezse devreye girer.

In [ ]:
!python src/train.py --model resnet50 --aug on --split data/splits.csv

In [ ]:
!python src/evaluate.py --model resnet50 --weights results/resnet50_best.pt --split test --write-csv
!cat results/resnet50_val.json
!cat results/results.csv

In [ ]:
!python src/kazanan.py
!cat results/kazanan.txt

In [ ]:
!zip -r /content/resnet50_sonuc.zip results/resnet50_best.pt results/resnet50_egri.png results/resnet50_val.json results/cm_resnet50.png results/results.csv results/kazanan.txt
from google.colab import files
files.download('/content/resnet50_sonuc.zip')

## 2. Kazananın artırmasız koşusu (protokol)

Model `results/kazanan.txt` içindeki addır. Tek fark `--aug off`'tur; başka hiçbir ayar değişmez.

In [ ]:
kazanan = Path('results/kazanan.txt').read_text(encoding='utf-8').strip()
assert kazanan in ('resnet50', 'efficientnet_b0'), kazanan
print('kazanan', kazanan)
!python src/train.py --model {kazanan} --aug off --split data/splits.csv

In [ ]:
!python src/evaluate.py --model {kazanan} --weights results/{kazanan}_aug_off.pt --split test --write-csv
!python src/sekiller.py
!cat results/results.csv

In [ ]:
!zip -r /content/aug_off_sonuc.zip results/{kazanan}_aug_off.pt results/{kazanan}_aug_off_egri.png results/{kazanan}_aug_off_val.json results/cm_{kazanan}_aug_off.png results/results.csv results/kazanan.txt results/sekil_macro_f1.png results/sekil_karisiklik.png
from google.colab import files
files.download('/content/aug_off_sonuc.zip')